In [66]:
import requests
import zipfile
import io
import os
import json
import re
import pandas as pd
import numpy as np
import joblib

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [1]:
import requests
import zipfile
import io

# URL of the dataset
url = "https://academy.hackthebox.com/storage/modules/292/skills_assessment_data.zip"
# Download the dataset
response = requests.get(url)
if response.status_code == 200:
    print("Download successful")
else:
    print("Failed to download the dataset")



Download successful


In [2]:
# Extract the dataset
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("imdb_dataset")
    print("Extraction successful")


Extraction successful


In [3]:
import os

# List the extracted files
extracted_files = os.listdir("imdb_dataset")
print("Extracted files:", extracted_files)


Extracted files: ['train.json', 'test.json']


In [8]:
import json

with open("imdb_dataset/train.json", "r") as f:
    data = json.load(f)

print(data[0])

{'text': 'Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High\'s satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers\' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I\'m here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn\'t!', 'label': 1}


In [9]:
import json
import pandas as pd

with open("imdb_dataset/train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(df.columns)

                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1
Index(['text', 'label'], dtype='str')


In [10]:
# Display basic information about the dataset
print("-------------------- HEAD --------------------")
print(df.head())
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())


-------------------- HEAD --------------------
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1
-------------------- DESCRIBE --------------------
             label
count  25000.00000
mean       0.50000
std        0.50001
min        0.00000
25%        0.00000
50%        0.50000
75%        1.00000
max        1.00000
-------------------- INFO --------------------
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    25000 non-null  str  
 1   label   25000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 32.0 MB
None


In [11]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())


Missing values:
 text     0
label    0
dtype: int64


In [12]:
# Check for duplicates
print("Duplicate entries:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates()


Duplicate entries: 96


In [13]:
import nltk

# Download the necessary NLTK data files
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("=== BEFORE ANY PREPROCESSING ===") 
print(df.head(5))


=== BEFORE ANY PREPROCESSING ===
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1


[nltk_data] Downloading package punkt to /home/beylessen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/beylessen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/beylessen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
# Convert all text text to lowercase
df["text"] = df["text"].str.lower()
print("\n=== AFTER LOWERCASING ===")
print(df["text"].head(5))



=== AFTER LOWERCASING ===
0    bromwell high is a cartoon comedy it ran at th...
1    homelessness or houselessness as george carlin...
2    brilliant overacting by lesley ann warren best...
3    this is easily the most underrated film inn th...
4    this is not the typical mel brooks film it was...
Name: text, dtype: str


In [18]:
import re

# Remove non-essential punctuation and numbers, keep useful symbols like $ and !
df["text"] = df["text"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
print("\n=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===")
print(df["text"].head(5))



=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===
0    bromwell high is a cartoon comedy it ran at th...
1    homelessness or houselessness as george carlin...
2    brilliant overacting by lesley ann warren best...
3    this is easily the most underrated film inn th...
4    this is not the typical mel brooks film it was...
Name: text, dtype: str


In [20]:
from nltk.tokenize import word_tokenize

# Split each text into individual tokens
df["text"] = df["text"].apply(word_tokenize)
print("\n=== AFTER TOKENIZATION ===")
print(df["text"].head(5))



=== AFTER TOKENIZATION ===
0    [bromwell, high, is, a, cartoon, comedy, it, r...
1    [homelessness, or, houselessness, as, george, ...
2    [brilliant, overacting, by, lesley, ann, warre...
3    [this, is, easily, the, most, underrated, film...
4    [this, is, not, the, typical, mel, brooks, fil...
Name: text, dtype: object


In [21]:
from nltk.corpus import stopwords

# Define a set of English stop words and remove them from the tokens
stop_words = set(stopwords.words("english"))
df["text"] = df["text"].apply(lambda x: [word for word in x if word not in stop_words])
print("\n=== AFTER REMOVING STOP WORDS ===")
print(df["text"].head(5))



=== AFTER REMOVING STOP WORDS ===
0    [bromwell, high, cartoon, comedy, ran, time, p...
1    [homelessness, houselessness, george, carlin, ...
2    [brilliant, overacting, lesley, ann, warren, b...
3    [easily, underrated, film, inn, brooks, cannon...
4    [typical, mel, brooks, film, much, less, slaps...
Name: text, dtype: object


In [22]:
from nltk.stem import PorterStemmer

# Stem each token to reduce words to their base form
stemmer = PorterStemmer()
df["text"] = df["text"].apply(lambda x: [stemmer.stem(word) for word in x])
print("\n=== AFTER STEMMING ===")
print(df["text"].head(5))



=== AFTER STEMMING ===
0    [bromwel, high, cartoon, comedi, ran, time, pr...
1    [homeless, houseless, georg, carlin, state, is...
2    [brilliant, overact, lesley, ann, warren, best...
3    [easili, underr, film, inn, brook, cannon, sur...
4    [typic, mel, brook, film, much, less, slapstic...
Name: text, dtype: object


In [25]:
# Rejoin tokens into a single string for feature extraction
df["text"] = df["text"].apply(lambda x: " ".join(x))
print("\n=== AFTER JOINING TOKENS BACK INTO STRINGS ===")
print(df["text"].head(5))



=== AFTER JOINING TOKENS BACK INTO STRINGS ===
0    bromwel high cartoon comedi ran time program s...
1    homeless houseless georg carlin state issu yea...
2    brilliant overact lesley ann warren best drama...
3    easili underr film inn brook cannon sure flaw ...
4    typic mel brook film much less slapstick movi ...
Name: text, dtype: str


In [68]:
vectorizer = TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1, 2), sublinear_tf=True)

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Labels (target variable)
y = df["label"]

In [69]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Build the pipeline by combining vectorization and classification
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", MultinomialNB())
])


In [71]:
# Define the parameter grid for hyperparameter tuning
param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

# Perform the grid search with 5-fold cross-validation and the F1-score as metric
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)

# Fit the grid search on the full dataset
grid_search.fit(df["text"], y)

# Extract the best model identified by the grid search
best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)


Best model parameters: {'classifier__alpha': 1.0}


In [39]:
new_texts = [
    "This movie was absolutely amazing! The story and acting were perfect.",
    "I really enjoyed the film, especially the soundtrack and cinematography.",
    "It was an okay movie, nothing special but not bad either.",
    "The movie was kind of boring and too long for my taste.",
    "I hated this film. The plot made no sense and the acting was terrible.",
    "What a masterpiece! One of the best movies I’ve ever seen.",
    "It was decent, but I expected much more from the ending.",
    "I loved the characters and the emotional depth of the story.",
    "The movie was disappointing. I almost fell asleep halfway through.",
    "Not bad, but definitely not something I would watch again.",
    "An outstanding performance by the lead actor, truly impressive.",
    "The storyline was confusing and hard to follow.",
    "Pretty good overall, some scenes were really enjoyable.",
    "I feel mixed about this movie — some parts were great, others not so much.",
    "Worst movie I’ve watched this year, total waste of time."
]

In [57]:
import numpy as np
import re

# Preprocess function that mirrors the training-time preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s$!]", "", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)


In [58]:
# Preprocess and vectorize texts
processed_texts = [preprocess_text(msg) for msg in new_texts]


In [59]:
# Transform preprocessed texts into feature vectors
X_new = best_model.named_steps["vectorizer"].transform(processed_texts)


In [60]:
# Predict with the trained classifier
predictions = best_model.named_steps["classifier"].predict(X_new)
prediction_probabilities = best_model.named_steps["classifier"].predict_proba(X_new)


In [72]:
# Display predictions and probabilities for each evaluated text
for i, msg in enumerate(new_texts):
    prediction = "Positive" if predictions[i] == 1 else "Negative"
    spam_probability = prediction_probabilities[i][1]  # Probability of being spam
    ham_probability = prediction_probabilities[i][0]   # Probability of being not spam
    
    print(f"text: {msg}")
    print(f"Prediction: {prediction}")
    print(f"Positive Probability: {spam_probability:.2f}")
    print(f"Negative Probability: {ham_probability:.2f}")
    print("-" * 50)


text: This movie was absolutely amazing! The story and acting were perfect.
Prediction: Positive
Positive Probability: 0.97
Negative Probability: 0.03
--------------------------------------------------
text: I really enjoyed the film, especially the soundtrack and cinematography.
Prediction: Positive
Positive Probability: 1.00
Negative Probability: 0.00
--------------------------------------------------
text: It was an okay movie, nothing special but not bad either.
Prediction: Negative
Positive Probability: 0.00
Negative Probability: 1.00
--------------------------------------------------
text: The movie was kind of boring and too long for my taste.
Prediction: Negative
Positive Probability: 0.03
Negative Probability: 0.97
--------------------------------------------------
text: I hated this film. The plot made no sense and the acting was terrible.
Prediction: Negative
Positive Probability: 0.00
Negative Probability: 1.00
--------------------------------------------------
text: What a

In [73]:
with open("imdb_dataset/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_df = pd.DataFrame(test_data)
test_df["text"] = test_df["text"].apply(preprocess_text)

test_preds = best_model.predict(test_df["text"])
print("Test accuracy:", accuracy_score(test_df["label"], test_preds))
print(classification_report(test_df["label"], test_preds))

Test accuracy: 0.8632
              precision    recall  f1-score   support

           0       0.85      0.88      0.87     12500
           1       0.88      0.84      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



In [63]:
import joblib
# Save the trained model to a file for future use
model_filename = 'sentiment_analyzer.joblib'
joblib.dump(best_model, model_filename)

print(f"Model saved to {model_filename}")

Model saved to sentiment_analyzer.joblib


In [46]:
loaded_model = joblib.load(model_filename)
predictions = loaded_model.predict(new_texts)